In [ ]:
# -------------------------------
# Cell 1: Imports
# -------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
import numpy as np

# -------------------------------
# Cell 2: Define HybridSN
# -------------------------------
class HybridSN(nn.Module):
    def __init__(self, in_bands, num_classes):
        super().__init__()
        self.conv3d_1 = nn.Conv3d(1, 8, kernel_size=(7,3,3), padding=(3,1,1))
        self.bn3d_1   = nn.BatchNorm3d(8)
        self.conv3d_2 = nn.Conv3d(8,16,kernel_size=(5,3,3), padding=(2,1,1))
        self.bn3d_2   = nn.BatchNorm3d(16)
        self.conv3d_3 = nn.Conv3d(16,32,kernel_size=(3,3,3), padding=(1,1,1))
        self.bn3d_3   = nn.BatchNorm3d(32)
        self.conv2d_1 = nn.Conv2d(32 * in_bands, 64, kernel_size=3, padding=1)
        self.bn2d_1   = nn.BatchNorm2d(64)
        self.conv2d_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2d_2   = nn.BatchNorm2d(64)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(64, 256)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, 128)
        self.drop2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        N, C, H, W = x.shape
        x = x.unsqueeze(1)  # (N,1,C,H,W)
        x = F.relu(self.bn3d_1(self.conv3d_1(x)))
        x = F.relu(self.bn3d_2(self.conv3d_2(x)))
        x = F.relu(self.bn3d_3(self.conv3d_3(x)))
        N, ch, D, H, W = x.shape
        x = x.permute(0,1,2,3,4).contiguous().view(N, ch*D, H, W)
        x = F.relu(self.bn2d_1(self.conv2d_1(x)))
        x = F.relu(self.bn2d_2(self.conv2d_2(x)))
        x = self.avgpool(x).view(N, -1)
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = self.fc3(x)
        return x

# -------------------------------
# Cell 3: Load checkpoint
# -------------------------------
checkpoint_path = "HybridSN_full_checkpoint.pth"  # path to your saved checkpoint
in_bands = 30     # adjust according to your dataset
num_classes = 16  # adjust according to your dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HybridSN(in_bands=in_bands, num_classes=num_classes).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Checkpoint loaded. Best val_acc:", checkpoint['val_acc'])

# -------------------------------
# Cell 4: Load test dataset
# -------------------------------
# Replace these with your actual test data
# X_test: shape (N_samples, in_bands, H, W)
# y_test: shape (N_samples,)

# Example using random data for template:
X_test = np.random.rand(100, in_bands, 25, 25).astype(np.float32)
y_test = np.random.randint(0, num_classes, size=(100,))

test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

# -------------------------------
# Cell 5: Evaluate on test set
# -------------------------------
y_true, y_pred = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        outputs = model(Xb)
        preds = outputs.argmax(dim=1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(yb.cpu().numpy())

acc = accuracy_score(y_true, y_pred)
print("Test Accuracy: {:.4f}".format(acc))

# -------------------------------
# Cell 6: Predict on new data
# -------------------------------
# Example: single input
x_new = np.random.rand(1, in_bands, 25, 25).astype(np.float32)
x_tensor = torch.tensor(x_new).to(device)

with torch.no_grad():
    output = model(x_tensor)
    pred_class = output.argmax(dim=1)
    print("Predicted class:", pred_class.item())

FileNotFoundError: [Errno 2] No such file or directory: 'HybridSN_full_checkpoint.pth'